# Training the stacked random forest

This notebook trains stage 2 of the model: a random forest that combines the vision model's
grade probabilities with patient data. It is provided for reference; *you do not need to run it*,
the trained model is in `models/stacker_rf.joblib`.

The random forest is trained only on patients marked `used_for == "stacker"` in `data/train.csv`.
The vision model never saw these images during its training, so its outputs on them are as
(un)reliable as on new patients. Its outputs on its own training images would be overconfident.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, cohen_kappa_score

from retinopathy_model import (CNN_FEATURES, FEATURES, GRADES, RetinopathyModel, load_image, load_patients,
                               stacker_features)

CNN_PATH = "model.h5"
IMAGE_DIR = "images"

## Data

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.used_for.value_counts())

patients_by_id = {p.id_code: p for p in load_patients()}
stacker_patients = [patients_by_id[i] for i in train[train.used_for == "stacker"].id_code]
test_patients = [patients_by_id[i] for i in test.id_code]
print(f"{len(stacker_patients)} patients for training, {len(test_patients)} for testing")

## Stage 1: vision model outputs

Runs the vision model on all training and test images (a few minutes on a CPU).

In [ ]:
model = RetinopathyModel(CNN_PATH, stacker_path=None)


def cnn_outputs(patients):
    images = [load_image(f"{IMAGE_DIR}/{p.id_code}.png") for p in patients]
    return np.concatenate([model.cnn_probabilities(images[i:i + 32]) for i in range(0, len(images), 32)])


X_train = stacker_features(cnn_outputs(stacker_patients), stacker_patients)
X_test = stacker_features(cnn_outputs(test_patients), test_patients)
y_train = np.array([p.diagnosis for p in stacker_patients])
y_test = np.array([p.diagnosis for p in test_patients])
X_train.head()

## Stage 2: random forest

In [ ]:
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, random_state=0, n_jobs=-1)
rf.fit(X_train[FEATURES], y_train)

## Evaluation on the test set

Compares the final model with the vision model alone. "Referable" retinopathy (grade 2 or higher)
should be referred to an eye specialist.

In [ ]:
def report(pred):
    referable, referable_true = pred >= 2, y_test >= 2
    return {"accuracy": accuracy_score(y_test, pred),
            "quadratic kappa": cohen_kappa_score(y_test, pred, weights="quadratic"),
            "referable sensitivity": (referable & referable_true).sum() / referable_true.sum(),
            "referable specificity": (~referable & ~referable_true).sum() / (~referable_true).sum()}


cnn_pred = X_test[CNN_FEATURES].to_numpy().argmax(axis=1)
pd.DataFrame({"vision model only": report(cnn_pred),
              "vision model + patient data": report(rf.predict(X_test[FEATURES]))}).round(3)

In [ ]:
grade_names = lambda grades, name: pd.Series(pd.Categorical.from_codes(grades, list(GRADES.values())), name=name)
pd.crosstab(grade_names(y_test, "clinician"), grade_names(rf.predict(X_test[FEATURES]), "model"))

## Save the model

In [ ]:
joblib.dump(rf, "models/stacker_rf.joblib", compress=3)